# Milestone 3 lab, notebook 6 of 8: Fix one day

## Where we are

| Notebook | What you do |
|---|---|
| 1. One order, one row | Write the same rows three ways: append, overwrite, upsert |
| 2. Which column is the key? | Merge on the wrong key. Then meet an order that arrives twice in one run |
| 3. Run it twice | Prove that a second run changes nothing |
| 4. At least once, exactly once | Crash the load in three places, and see what each one costs |
| 5. The update that came late | Find a change the watermark has passed. Catch it with a lookback |
| **6. Fix one day** (you are here) | Break Friday's numbers, and rebuild Friday alone |
| 7. The order that vanished | A deleted order, and why the query cannot see it |
| 8. The production version | The three properties, as tests |

Finance sometimes needs one old day put right: a rule was wrong, or someone edited a table by hand. They want to fix **that day**, and be sure no other day moved. This notebook breaks Friday, and repairs Friday alone.

In [1]:
import psycopg

from lab_helpers import lab_warehouse, run_pipeline, show_doc, table

con = lab_warehouse(fresh=True)
run_pipeline(con, "2026-06-08", "2026-06-09")
print("Two mornings are loaded.")

Two mornings are loaded.


## Step 1: The report is a table now

In milestone 2 the report was a view: it was worked out again each time someone read it. Now it is a real table. Each row is one business day in one city.

In [2]:
def report(days="'2026-06-04', '2026-06-05', '2026-06-06'"):
    return table(con, f"""
        SELECT order_date, city, orders, cancelled_orders, revenue_cents, refunded_cents
        FROM analytics.daily_orders WHERE order_date IN ({days}) ORDER BY order_date, city
    """)

right = report()
right

,order_date,city,orders,cancelled_orders,revenue_cents,refunded_cents
0,2026-06-04,LKS,12,0,23922,0
1,2026-06-04,NGT,9,0,20101,0
2,2026-06-04,RVB,9,0,15175,0
3,2026-06-05,LKS,11,0,32381,0
4,2026-06-05,NGT,15,0,34553,578
5,2026-06-05,RVB,11,0,30170,0
6,2026-06-06,LKS,9,0,20078,0
7,2026-06-06,NGT,17,0,39599,0
8,2026-06-06,RVB,7,0,18994,0


All the rows of one day belong together. Such a slice has a name: a **partition**. Here, one business day is one partition.

## Step 2: Break Friday

Someone "fixes" a number by hand, and deletes a row by accident.

In [3]:
con.execute("UPDATE analytics.daily_orders SET revenue_cents = 1 WHERE order_date = '2026-06-05' AND city = 'NGT'")
con.execute("DELETE FROM analytics.daily_orders WHERE order_date = '2026-06-05' AND city = 'RVB'")
report("'2026-06-05'")

,order_date,city,orders,cancelled_orders,revenue_cents,refunded_cents
0,2026-06-05,LKS,11,0,32381,0
1,2026-06-05,NGT,15,0,1,578


## Step 3: Replace one day, by hand

The clean orders table still holds the truth. So throw Friday's report rows away, and build them again from the orders. Both steps go in one transaction, so nobody ever sees Friday half built.

In [4]:
REBUILD_ONE_DAY = """
    INSERT INTO analytics.daily_orders
    SELECT order_date, city,
           count(*) FILTER (WHERE status <> 'cancelled'),
           count(*) FILTER (WHERE status = 'cancelled'),
           coalesce(sum(total_cents - refunded_cents) FILTER (WHERE status <> 'cancelled'), 0),
           coalesce(sum(refunded_cents) FILTER (WHERE status <> 'cancelled'), 0)
    FROM core.orders
    WHERE order_date = %(day)s AND NOT is_deleted
    GROUP BY order_date, city
"""

def replace_day(day):
    with con.transaction():
        con.execute("DELETE FROM analytics.daily_orders WHERE order_date = %(day)s", {"day": day})
        con.execute(REBUILD_ONE_DAY, {"day": day})

replace_day("2026-06-05")
report("'2026-06-05'")

,order_date,city,orders,cancelled_orders,revenue_cents,refunded_cents
0,2026-06-05,LKS,11,0,32381,0
1,2026-06-05,NGT,15,0,34553,578
2,2026-06-05,RVB,11,0,30170,0


In [5]:
print("Friday, and the days around it, are exactly as before the damage:", report().equals(right))

Friday, and the days around it, are exactly as before the damage: True


Friday is right again. Thursday and Saturday were never touched: the `DELETE` and the `INSERT` both say `WHERE order_date = Friday`.

Use the same day in both. If you forget the `WHERE` in the `DELETE`, it removes every day in the table.

## Step 4: Is it safe to run twice?

Ask the question of every write. Replace Friday three more times.

In [6]:
for _ in range(3):
    replace_day("2026-06-05")
print("Still exactly as before:", report().equals(right))

Still exactly as before: True


Each time the day is deleted and built again, so it ends the same. Replacing a partition is idempotent.

### Compare it with its opposite

What if the job only inserted Friday's rows, without deleting first? Try it.

In [7]:
try:
    con.execute(REBUILD_ONE_DAY, {"day": "2026-06-05"})
except psycopg.Error as error:
    print("Postgres refused:")
    print(" ", error.diag.message_primary)

Postgres refused:
  duplicate key value violates unique constraint "daily_orders_pkey"


The report table has a primary key: one row per day and city. So the database refuses to hold Friday twice.

Without that key the insert would have succeeded, and Friday would simply have doubled. See it on a copy of the table with no key.

In [8]:
con.execute("CREATE TEMP TABLE report_without_key AS SELECT * FROM analytics.daily_orders")
con.execute(REBUILD_ONE_DAY.replace("analytics.daily_orders", "report_without_key"), {"day": "2026-06-05"})

table(con, """
    SELECT 'with delete + insert' AS how, sum(orders) AS friday_orders, sum(revenue_cents) AS friday_cents
    FROM analytics.daily_orders WHERE order_date = '2026-06-05'
    UNION ALL
    SELECT 'insert only, no key', sum(orders), sum(revenue_cents)
    FROM report_without_key WHERE order_date = '2026-06-05'
""")

,how,friday_orders,friday_cents
0,with delete + insert,37,97104
1,"insert only, no key",74,194208


That doubled row is exactly what happened at Arden in the story at the start of this project.

## Step 5: The pipeline does this every morning

Each run rebuilds the report for the business days of the orders it touched, and no others.

In [9]:
result = run_pipeline(con, "2026-06-10")
print("Run", result.run_id, "rebuilt these days:", ", ".join(result.days_rebuilt))

Run 2026-06-10_1 rebuilt these days: 2026-06-05, 2026-06-06, 2026-06-07, 2026-06-08, 2026-06-09


And a person can rebuild one day from a terminal, in this milestone's folder:

```
uv run python -m daily_orders --rebuild-day 2026-06-05
```

Doing a past day again has a name: **reprocessing**. Here you reprocessed one day of the report from the orders table. In milestone 5 you will reprocess whole runs for past days, which is called a backfill.

In [10]:
show_doc("LLD.md", "7. Rebuild days: replace one day")

**From `docs/LLD.md`**

> ## 7. Rebuild days: replace one day
> 
> ```sql
> BEGIN;
> DELETE FROM analytics.daily_orders WHERE order_date = ANY(:days);
> 
> INSERT INTO analytics.daily_orders
> SELECT order_date, city, count(*) FILTER (WHERE status <> 'cancelled'), ...
> FROM core.orders
> WHERE order_date = ANY(:days) AND NOT is_deleted
> GROUP BY order_date, city;
> COMMIT;
> ```
> 
> `:days` are the business days of the orders this run touched. Use the same days in the `DELETE` and in the `INSERT`.
> 
> To fix one day by hand:
> 
> ```sh
> uv run python -m daily_orders --rebuild-day 2026-06-05
> ```

In [11]:
con.close()

## Two new words

| Word | Meaning | Here |
|---|---|---|
| **Partition** | "A partition is one slice of a table." Here, all the rows of one business day | Friday 5 June: three rows, one per city |
| **Reprocessing** | Building a past slice again, to correct it | Delete Friday's rows and insert them again, in one transaction |

The pattern itself is called **replace one day**: an overwrite of one partition, not of the whole table.

The definitions are from the D8Loop chapters [Incremental Processing & Backfills](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/incremental-processing-and-backfills) and [Loading & Write Patterns](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/loading-and-write-patterns).

## What you now know

- One business day is one partition of the report.
- Replacing a day is a delete and an insert for that day, in one transaction.
- It corrects that day, touches no other, and is safe to run twice.
- An insert without the delete would double the day.

**Say it in an interview:** "My report table is partitioned by business day, and I write it with delete-and-insert per day in one transaction. So I can re-run last Friday alone, any number of times, and only Friday changes. I have a test that corrupts one day, rebuilds it, and checks that every other day is untouched."

Next: [notebook 7](07_the_order_that_vanished.ipynb) meets an order that disappears from the source.